In [3]:
from pathlib import Path # Python's standard library module for working with filesystem paths in an object-oriented way
from openai import OpenAI 
from dotenv import load_dotenv # Load environment variable
from pydantic import BaseModel, Field # For Domain models
from chromadb import PersistentClient # Chroma database
from tqdm import tqdm

### Package - Litellm 

The litellm library is a Python package that gives you a unified interface for calling many different large language model (LLM) providers — OpenAI, Anthropic, Azure, Cohere, Hugging Face, Google's Gemini/Vertex AI, AWS Bedrock, Ollama, and dozens more — all through the same function signature.

### Function - completion() 

The completion() function specifically mimics the OpenAI ChatCompletion API format. So instead of learning each provider's own SDK, you call:

```bash
response = completion(
    model="gpt-4",  # or "claude-sonnet-4-6", "gemini/gemini-pro", etc.
    messages=[{"role": "user", "content": "Hello!"}]  
)
```


In [4]:
from litellm import completion
import numpy as np

### TSNE 
TSNE is a class in scikit-learn's sklearn.manifold module, used for t-distributed Stochastic Neighbor Embedding — a technique for dimensionality reduction, mainly used for visualizing high-dimensional data.

In [5]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [6]:
# Load environment variables
load_dotenv(override=True)

# Model name
MODEL = "gpt-4.1-nano"

# Vector database and related details
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"

# Knowledge source path
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

In [7]:
# Get a handle of OpenAI object
openai = OpenAI()

In [8]:
class Result(BaseModel):
    page_content: str
    metadata : dict

In [9]:
# Class to represent a Chunk
class Chunk (BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    originial_text : str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    '''
    document is a dict
    metadata is a dict
    '''

    def as_result(self, document):
        metadata = { 
            "source" : document["source"],
            "type" : document["type"]
        }
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.originial_text, metadata=metadata)

class Chunks (BaseModel):
    chunks : list[Chunk]

### Step 1: Fetch documents from the knowledge base.

In [10]:
def fetch_documents():
    documents = [] # A plain list that will collect list of dictionaries with one dictionary per file loaded.

    for folder in KNOWLEDGE_BASE_PATH.iterdir(): # Yields everything directly inside it — both files and subfolders.
        doc_type = folder.name
        ''' .rglob("*.md") recursively searches folder (including any nested subfolders) 
        for files matching the pattern *.md — i.e., all Markdown files,
        no matter how deeply nested.
        '''
        for file in folder.rglob("*.md"):  
            '''
            Opens the file in text mode with UTF-8 encoding 
            (important for handling non-ASCII characters correctly).
            '''
            with open(file, "r", encoding="utf-8") as f:
                '''
                file.as_posix() converts the Path object to a forward-slash-style string path 
                (e.g. "knowledge_base/faq/a.md"), regardless of the OS — this keeps source paths 
                consistent even on Windows.
                
                Appends a dictionary with three keys: type (the category from the folder name), 
                source (the file path), and text (the file's contents).
                '''
                documents.append({"type" : doc_type, "source" : file.as_posix(), "text" : f.read()})
    
    print(f"Loaded {len(documents)} documents")
    return documents

In [11]:
documents = fetch_documents()

Loaded 76 documents


In [12]:
documents[0]["text"]

"# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integrations\nRellm's architecture is designed for effortless integration with exi

### Step 2: Create the chunks

In [13]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
        You take a document and you split the document into overlapping chunks for a KnowledgeBase.

        The document is from the shared drive of a company called Insurellm.
        The document is of type: {document["type"]}
        The document has been retrieved from: {document["source"]}

        A chatbot will use these chunks to answer questions about the company.
        You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
        This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
        There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

        For each chunk, you should provide a headline, a summary, and the original text of the chunk.
        Together your chunks should represent the entire document with overlap.

        Here is the document:

        {document["text"]}

        Respond with the chunks.
        """

In [ ]:
print(make_prompt(documents[0]))

In [14]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [15]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\n        You take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\n        The document is from the shared drive of a company called Insurellm.\n        The document is of type: products\n        The document has been retrieved from: knowledge-base/products/Rellm.md\n\n        A chatbot will use these chunks to answer questions about the company.\n        You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\n        This document should probably be split into 8 chunks, but you can have more or less as appropriate.\n        There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\n        For each chunk, you should provide a headline, a summary, and the original text of the chunk.\n        Together your chunks 

In [16]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]


In [28]:
process_document(documents[0])

[Result(page_content='Product Overview of Rellm\n\nRellm is an enterprise reinsurance platform developed by Insurellm that integrates AI-driven analytics, seamless system integrations, risk assessment, customizable dashboards, and compliance tools to optimize reinsurance operations.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': 'knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="Fea

In [17]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [18]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [49:54<00:00, 39.40s/it]   


In [20]:
print(len(chunks))

396


In [21]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [22]:
create_embeddings(chunks)

Vectorstore created with 396 documents


In [23]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [24]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [25]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Re-ranking

In [26]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
        )

In [27]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [28]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [29]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [30]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

2023 Performanc...
Career Progress...
Additional HR N...
Performance Hig...
Performance and...
Performance His...
Career Progress...
Performance His...
Performance Rev...
Performance His...


In [31]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [32]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

2023 Performanc...
Career Progress...
Additional HR N...
Performance Hig...
Performance and...
Performance His...
Career Progress...
Performance His...
Performance Rev...
Performance His...


In [33]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [34]:
reranked = rerank(question, chunks)

[1, 8, 11, 12, 20, 9, 13, 10, 14, 2, 3, 4, 5, 6, 7, 15, 16, 17, 18, 19]


In [35]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [36]:
reranked[0].page_content

'Insurellm Career Progression (Part 1)\n\nDetails Samuel Trenton\'s career path at Insurellm, highlighting his progression from Junior Data Analyst to Senior Data Scientist, with notable achievements.\n\n## Insurellm Career Progression\n- **January 2020 - Present:** Senior Data Scientist  \n  *Promoted for demonstrating exceptional analytical skills and leadership potential. Led several projects that improved customer segmentation strategies, resulting in a 15% increase in customer retention.*\n\n- **June 2018 - December 2019:** Data Scientist  \n  *Joined the Insurellm team and worked on developing predictive modeling techniques to assess risk for both B2B and B2C customers. Received recognition for the success of the "Risk Assessment Model" project.*\n\n- **August 2016 - May 2018:** Junior Data Analyst  \n  *Started at Insurellm as a Junior Data Analyst, focusing on data cleaning and preliminary analysis of customer data. Received training in various data visualization techniques, wh

In [37]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [38]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [39]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [40]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [41]:
rewrite_query("Who won the IIOTY award?", [])

'Who received the IIOTY award?'

In [42]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [43]:
answer_question("Who won the IIOTY award?", [])

Who is the recipient of the IIOTY award?
[1, 2, 3, 10, 11, 4, 7, 9, 8, 14, 13, 15, 12, 5, 6, 16, 17, 18, 19, 20]


('Maxine Thompson received the IIOTY Innovator Award in 2023.',
 [Result(page_content='2023 Performance and Leadership\n\nIn 2023, Maxine exceeded expectations, took on mentoring roles, led data governance initiatives, and received the IIOTY Innovator Award.\n\n- **2023**: *Exceeds Expectations*  \n Maxine has taken on mentoring responsibilities and is leading a cross-functional team for data governance initiatives, showcasing her leadership and solidifying her role at Insurellm.', metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'type': 'employees'}),
  Result(page_content='Career Progression at Insurellm - Part 3\n\nSince 2021, Maxine has been a Senior Data Engineer, leading strategic projects, mentoring, and receiving recognition such as the IIOTY 2023 award.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now ment

In [44]:
answer_question("Who went to Manchester University?", [])

Who is an alumnus of Manchester University at Insurellm?
[1, 4, 13, 14, 16, 18, 20, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 15, 17, 19]


('Based on the information available, there is no record of any individual associated with Insurellm having attended Manchester University.',
 [Result(page_content='Introduction and Founding of Insurellm\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to innovate within the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'type': 'company', 'source': 'knowledge-base/company/about.md'}),
  Result(page_content="Career Progression at Insurellm and Previous Roles\n\nThis section details Avery Lancaster's career progression, from her role as Business Analyst to her founding of Insurellm and her previous positions.\n\n## Insurellm Career Progression\n-